# Classifier Performance Analysis 
How does test accuracy degrade under different threat models.
Classifier Training set covers all T3 attacks (incl.ALIE, IPM)

In [1]:
%load_ext autoreload
import sys
import copy
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from pathlib import Path
from fastnanoid import generate
from datetime import datetime

In [2]:
project_root = Path().resolve().parent
sys.path.insert(0, str(project_root))
lib_path = project_root / "lib"
sys.path.insert(0, str(lib_path))        
sys.path.insert(0, str(project_root))   

In [3]:
%autoreload 2
from lib.graph_factory import GraphFactory
from lib.proj_const_estimator import ProjConstEstimator
from lib.utils import fetch_dataset, get_alphas, rng, seed
from lib.classifier import ByzClassifier
from lib.system import SystemSimulator
from lib.config import BASE_CONF, LOGFILE, NUM_NODES

In [4]:
RUN_DIR = Path().resolve()
config = BASE_CONF
config['sys']['atk_type'] = 'ALIE'
atks = ['label_flip', 'sign_flip', 'gaussian', 'ALIE', 'IPM']

In [5]:
global_dataset = fetch_dataset('MNIST')
gf = GraphFactory(config['train']['num_nodes'], config['train']['b'], seed('graph'))
proj_const_estimator = ProjConstEstimator(config, global_dataset, rng('proj-const-estimator')) 
classifier = ByzClassifier(config, global_dataset, rng('byz-classifier'))
sys_sim = SystemSimulator(config, global_dataset, rng('ml-system-sim'), LOGFILE)

In [6]:
proj_const_estimator.configure(gf,config)
proj_const = proj_const_estimator.estimate()

In [7]:
run_results = dict()
atks = ['label_flip', 'sign_flip', 'gaussian', 'ALIE', 'IPM']
local_seed = seed('experiment-7')
classifier.configure(gf, config, proj_const) 

In [8]:
train_sim_data = dict()
train_sim_seed = local_seed
for atk in atks:
    print(f'Train simulating {atk}')
    X, y = classifier.simulate(atk, train_sim_seed)
    train_sim_data[atk] = { 'X': X, 'y': y}

Train simulating label_flip
Train simulating sign_flip
Train simulating gaussian
Train simulating ALIE
Train simulating IPM


In [9]:
test_sim_data = dict()
test_sim_seed = local_seed + 17
for atk in atks:
    print(f'Test simulating {atk}')
    X, y = classifier.simulate(atk, test_sim_seed)
    test_sim_data[atk] = { 'X': X, 'y': y}

Test simulating label_flip
Test simulating sign_flip
Test simulating gaussian
Test simulating ALIE
Test simulating IPM


In [10]:
rows = []
for train_atk in atks:
    for test_atk in atks:
        if(train_atk != test_atk):
            RUN_ID = generate()
            print(train_atk, test_atk)
            
            X_train = train_sim_data[train_atk]['X']
            y_train = train_sim_data[train_atk]['y']
    
            X_test = test_sim_data[test_atk]['X']
            y_test = test_sim_data[test_atk]['y']
    
            classifier.init_preproc()
            X_tr = classifier.feat_pre_proc.fit_transform(X_train)
            C_specs = classifier.train_and_eval(X_tr, y_train, quantile=0.8)
    
            X_te = classifier.feat_pre_proc.transform(X_test)
            
            run_results = classifier.test(X_te, y_test)
            run_results['proj_const'] = proj_const  
            run_results['train_atk'] = train_atk
            run_results['test_atk'] = test_atk
            run_results['id'] = RUN_ID
            rows.append(run_results)
            
df = pd.DataFrame(rows)
df.set_index('id', inplace=True)

label_flip sign_flip
label_flip gaussian
label_flip ALIE
label_flip IPM
sign_flip label_flip
sign_flip gaussian
sign_flip ALIE
sign_flip IPM
gaussian label_flip
gaussian sign_flip
gaussian ALIE
gaussian IPM
ALIE label_flip
ALIE sign_flip
ALIE gaussian
ALIE IPM
IPM label_flip
IPM sign_flip
IPM gaussian
IPM ALIE


In [11]:
for atk in atks:
    RUN_ID = generate()
    config['train']['train_atks'] = [atk]
    config['train']['val_atks'] = [atk]
    X_train, y_train, X_test, y_test = classifier.run_simulations() # handles feature transforms
    C_specs = classifier.train_and_eval(X_train, y_train, quantile=0.8)
    run_results = classifier.test(X_test, y_test)
    run_results['proj_const'] = proj_const  
    run_results['train_atk'] =  atk
    run_results['test_atk'] = atk
    df[RUN_ID] = run_results
    


KeyboardInterrupt



In [ ]:
df